# Instacart Gold ML and hyperparameter tuning

Train models on the **shopping-behavior** Gold worksheet in Google Sheets.
Predict the historical aggregate `orders` for a day-of-week/hour combination using only those two inputs.

**Start here:** Open in Google Colab (CPU is enough), run setup, paste your Google Sheet URL in configuration, and run the remaining cells in order. The default public mode reads a sheet shared as Anyone with the link / Viewer without credentials. For a private sheet, select google_sheets and sign in with an account that can read it. An Excel/CSV upload option is also included.

This notebook includes validation, exploration, a mean baseline, Ridge regression, Random Forest, optional randomized hyperparameter search, untouched test evaluation, scenario estimates, and downloadable models/results.

The supplied workbook contains only **168 aggregate rows**, not 168 independent days. The model estimates patterns within this historical snapshot; it cannot forecast tomorrow/monthly demand, individual purchases, or revenue. Actual aggregate values are preferable when already known. For forecasting, collect dated observations and use a chronological evaluation.


## 1. Install dependencies
Run this cell first. If Colab asks for a runtime restart after installation, restart and continue from imports.


In [ ]:
%pip install -q "scikit-learn==1.7.2" "gspread>=6,<7" "pandas>=2.2,<3" "numpy>=1.26,<3" "matplotlib>=3.8,<4" "openpyxl>=3.1,<4" "joblib>=1.4,<2"


In [ ]:
import json
import hashlib
import platform
import shutil
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_validate, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('Python:', platform.python_version(), '| scikit-learn:', sklearn.__version__)


## 2. Configure the run

Use the exact **shopping-behavior** worksheet name, not the KPI summary or top-products tab.
Set `RUN_TUNING = False` for a quick first pass. Set it to `True` to search Random Forest settings.
Tuning here means selecting ML hyperparameters, not fine-tuning an LLM. Change settings before inspecting the test results; repeatedly changing them after seeing test scores invalidates the holdout.


In [ ]:
DATA_SOURCE = 'public_google_sheets'  # public_google_sheets, google_sheets (private), or upload
GOOGLE_SHEET_URL = ''  # Paste https://docs.google.com/spreadsheets/d/.../edit here
WORKSHEET_NAME = 'shopping-behavior'
UPLOAD_PATH = ''  # Leave blank to show Colab upload dialog; accepts .xlsx or .csv
RUN_TUNING = True
N_ITER = 20
RANDOM_STATE = 42
TEST_FRACTION = 0.20
OUTPUT_DIR = Path('instacart_ml_outputs')


## 3. Load your data
Google Sheets access is read-only. This notebook does not alter the dashboard or its source sheets.


In [ ]:
if DATA_SOURCE == 'public_google_sheets':
    from urllib.parse import quote
    import re
    match = re.match(r'https://docs.google.com/spreadsheets/d/([a-zA-Z0-9_-]+)', GOOGLE_SHEET_URL)
    if not match:
        raise ValueError('Paste your Google Sheet URL in configuration first.')
    csv_url = f'https://docs.google.com/spreadsheets/d/{match.group(1)}/gviz/tq?tqx=out:csv&sheet={quote(WORKSHEET_NAME, safe="")}'
    try:
        raw_df = pd.read_csv(csv_url)
    except Exception as exc:
        raise RuntimeError('Public read failed. Set Anyone with the link to Viewer, or select google_sheets for sign-in, or upload an Excel/CSV export.') from exc
elif DATA_SOURCE == 'google_sheets':
    if not GOOGLE_SHEET_URL.startswith('https://docs.google.com/spreadsheets/d/'):
        raise ValueError('Paste your Google Sheet URL in the configuration cell first.')
    from google.colab import auth
    from google.auth import default
    import gspread
    auth.authenticate_user()
    credentials, _ = default(scopes=['https://www.googleapis.com/auth/spreadsheets.readonly'])
    client = gspread.authorize(credentials)
    worksheet = client.open_by_url(GOOGLE_SHEET_URL).worksheet(WORKSHEET_NAME)
    values = worksheet.get_all_values(value_render_option='UNFORMATTED_VALUE')
    if len(values) < 2:
        raise ValueError('The worksheet must have a header and data rows.')
    raw_df = pd.DataFrame(values[1:], columns=values[0])
elif DATA_SOURCE == 'upload':
    if not UPLOAD_PATH:
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError('Upload exactly one Excel workbook or CSV.')
        UPLOAD_PATH = next(iter(uploaded))
    suffix = Path(UPLOAD_PATH).suffix.lower()
    if suffix == '.xlsx':
        raw_df = pd.read_excel(UPLOAD_PATH, sheet_name=WORKSHEET_NAME)
    elif suffix == '.csv':
        raw_df = pd.read_csv(UPLOAD_PATH)
    else:
        raise ValueError('Use an .xlsx workbook or a shopping-behavior .csv export.')
else:
    raise ValueError('DATA_SOURCE must be public_google_sheets, google_sheets, or upload.')
print('Loaded shape:', raw_df.shape)
display(raw_df.head())


## 4. Validate the modeling table

Each row must represent one unique `(order_dow, order_hour_of_day)` pair.
Duplicate snapshots are rejected, because mixing repeated aggregates would contaminate evaluation.
Only day and hour become features. Same-row `items_ordered`, `unique_customers`, basket size, reordered items, and rates are excluded because they are observed alongside or derived from the target.


In [ ]:
INPUT_COLUMNS = ['order_dow', 'order_hour_of_day']
TARGET = 'orders'

def validate_data(frame):
    frame = frame.copy()
    frame.columns = [str(c).strip() for c in frame.columns]
    if frame.columns.duplicated().any():
        raise ValueError('Duplicate column headers found.')
    required = INPUT_COLUMNS + [TARGET]
    missing = set(required) - set(frame.columns)
    if missing:
        raise ValueError(f'Missing columns: {sorted(missing)}. Select shopping-behavior.')
    frame = frame.replace(r'^\s*$', np.nan, regex=True).dropna(how='all')
    df = frame[required].apply(pd.to_numeric, errors='coerce')
    if not np.isfinite(df.to_numpy(dtype=float)).all():
        raise ValueError('Required values must be numeric, finite and non-empty. Use unformatted numeric cells.')
    if (df != np.floor(df)).any().any():
        raise ValueError('Day, hour and order counts must be integers.')
    if not df.order_dow.between(0, 6).all() or not df.order_hour_of_day.between(0, 23).all():
        raise ValueError('Expected order_dow 0..6 and order_hour_of_day 0..23.')
    if (df.orders < 0).any() or df.orders.nunique() < 2:
        raise ValueError('Orders must be nonnegative and vary across rows.')
    if df.duplicated(INPUT_COLUMNS).any():
        raise ValueError('Duplicate day/hour pairs: select a single Gold snapshot.')
    if len(df) < 70 or df.order_dow.nunique() != 7 or df.order_hour_of_day.nunique() < 12:
        raise ValueError('Need at least 70 rows, all 7 day codes, and at least 12 hours for this exercise.')
    return df.astype('int64').sort_values(INPUT_COLUMNS).reset_index(drop=True)

df = validate_data(raw_df)
data_fingerprint = hashlib.sha256(df.to_csv(index=False).encode()).hexdigest()
print(f'Validated {len(df)} unique day/hour rows (full grid: 168).')
print('Data fingerprint:', data_fingerprint[:16])
if len(df) != 168:
    print('Partial grid: missing combinations are not assumed to be zero.')
display(df.describe())


## 5. Explore the historical pattern
Day codes retain the source encoding. Processing timestamps are not order dates.


In [ ]:
heatmap = df.pivot(index='order_dow', columns='order_hour_of_day', values='orders')
fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(heatmap, aspect='auto', cmap='Blues')
ax.set_xticks(range(len(heatmap.columns)), heatmap.columns)
ax.set_yticks(range(len(heatmap.index)), heatmap.index)
ax.set(xlabel='Hour of day', ylabel='Day code', title='Historical aggregate orders')
fig.colorbar(im, ax=ax, label='Orders in the entire snapshot')
plt.tight_layout()
plt.show()


## 6. Features and holdout

Encode hour and day with sine/cosine terms so midnight is close to 23:00.
Hold out complete hour groups across all days; no held-out hour enters model selection.
Four-fold cross-validation also groups by hour inside the training partition.
This tests estimation at omitted hours in the same snapshot, not future performance.


In [ ]:
def make_features(frame):
    day = pd.to_numeric(frame['order_dow'], errors='raise')
    hour = pd.to_numeric(frame['order_hour_of_day'], errors='raise')
    if day.isna().any() or hour.isna().any() or not day.between(0, 6).all() or not hour.between(0, 23).all():
        raise ValueError('Day must be 0..6 and hour 0..23, without missing values.')
    if (day != np.floor(day)).any() or (hour != np.floor(hour)).any():
        raise ValueError('Day and hour must be integers.')
    return pd.DataFrame({
        'day_sin': np.sin(2 * np.pi * day / 7),
        'day_cos': np.cos(2 * np.pi * day / 7),
        'hour_sin': np.sin(2 * np.pi * hour / 24),
        'hour_cos': np.cos(2 * np.pi * hour / 24),
        'hour_sin_2': np.sin(4 * np.pi * hour / 24),
        'hour_cos_2': np.cos(4 * np.pi * hour / 24),
    }, index=frame.index)

X, y = make_features(df), df[TARGET]
groups = df.order_hour_of_day
splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
train_groups = groups.iloc[train_idx]
assert set(train_groups).isdisjoint(set(groups.iloc[test_idx]))
cv = GroupKFold(n_splits=4)
print('Train rows:', len(train_idx), '| Test rows:', len(test_idx))
print('Held-out hours:', sorted(groups.iloc[test_idx].unique().tolist()))


## 7. Compare untuned models on training folds

MAE measures the typical absolute error in aggregate order counts (lower is better).
The mean baseline predicts the training mean regardless of inputs. The test set stays untouched here.


In [ ]:
models = {
    'Mean baseline': DummyRegressor(strategy='mean'),
    'Ridge regression': make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    'Random Forest': RandomForestRegressor(n_estimators=200, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=1),
}
cv_rows = []
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, groups=train_groups, cv=cv,
                            scoring='neg_mean_absolute_error', n_jobs=1, error_score='raise')
    losses = -scores['test_score']
    cv_rows.append({'model': name, 'cv_mae': losses.mean(), 'fold_mae_std': losses.std()})
cv_results = pd.DataFrame(cv_rows).sort_values('cv_mae').reset_index(drop=True)
display(cv_results.round(2))


## 8. Optional hyperparameter tuning

With `RUN_TUNING=True`, try `N_ITER` combinations of tree count, depth, minimum leaf size, and feature sampling. The best settings minimize training-fold MAE. Tuning may fail to improve performance; the notebook retains the best model based on CV, including the baseline.


In [ ]:
search = None
if RUN_TUNING:
    search = RandomizedSearchCV(
        RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1),
        param_distributions={
            'n_estimators': [100, 200, 400],
            'max_depth': [3, 5, 8, None],
            'min_samples_leaf': [1, 2, 4, 8],
            'max_features': [0.7, 1.0],
        },
        n_iter=N_ITER, scoring='neg_mean_absolute_error', cv=cv,
        random_state=RANDOM_STATE, n_jobs=1, refit=True, error_score='raise',
    )
    search.fit(X_train, y_train, groups=train_groups)
    models['Tuned Random Forest'] = search.best_estimator_
    cv_rows.append({'model': 'Tuned Random Forest', 'cv_mae': -search.best_score_,
                    'fold_mae_std': search.cv_results_['std_test_score'][search.best_index_]})
    print('Best hyperparameters:', search.best_params_)
else:
    print('Tuning skipped. Set RUN_TUNING=True and rerun from configuration to enable.')
cv_results = pd.DataFrame(cv_rows).sort_values('cv_mae').reset_index(drop=True)
selected_name = cv_results.iloc[0]['model']
selected_model = clone(models[selected_name]).fit(X_train, y_train)
display(cv_results.round(2))
print('Selected using training CV only:', selected_name)


## 9. Evaluate once on held-out hours

The selected model is fixed before this cell. Test metrics are reported for it and the baseline only.
R² can be negative; that means performance is worse than a constant test-mean reference.
WAPE is total absolute error divided by total actual orders. These metrics are not production guarantees.


In [ ]:
def metrics(actual, predicted):
    denominator = np.abs(actual).sum()
    return {'MAE': mean_absolute_error(actual, predicted),
            'RMSE': np.sqrt(mean_squared_error(actual, predicted)),
            'R2': r2_score(actual, predicted),
            'WAPE_pct': 100 * np.abs(actual - predicted).sum() / denominator if denominator else np.nan}

test_prediction = selected_model.predict(X_test)
baseline = DummyRegressor(strategy='mean').fit(X_train, y_train)
baseline_prediction = baseline.predict(X_test)
test_results = pd.DataFrame([
    {'model': 'Mean baseline', **metrics(y_test, baseline_prediction)},
    {'model': selected_name, **metrics(y_test, test_prediction)},
])
display(test_results.round(3))
if mean_absolute_error(y_test, test_prediction) >= mean_absolute_error(y_test, baseline_prediction):
    print('Selected model did not beat the baseline on held-out hours. Report this result as-is.')
test_predictions = df.iloc[test_idx].copy()
test_predictions['predicted_orders'] = test_prediction
test_predictions['absolute_error'] = np.abs(y_test.to_numpy() - test_prediction)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(y_test, test_prediction, alpha=0.8)
limits = [min(y_test.min(), test_prediction.min()), max(y_test.max(), test_prediction.max())]
axes[0].plot(limits, limits, '--', color='gray')
axes[0].set(xlabel='Actual orders', ylabel='Predicted orders', title='Untouched test hours')
axes[1].scatter(test_prediction, y_test.to_numpy() - test_prediction)
axes[1].axhline(0, linestyle='--', color='gray')
axes[1].set(xlabel='Predicted orders', ylabel='Actual minus predicted', title='Residuals')
plt.tight_layout()
plt.show()


## 10. Refit and estimate a day/hour pattern

After evaluation, refit the selected configuration on all rows for demonstration.
Its fitted predictions are not test results. The estimate is a total over the original
snapshot, not an expected number of orders next Monday. Do not change snapshot size
and compare predicted totals as if they had the same exposure.


In [ ]:
final_model = clone(models[selected_name]).fit(X, y)
DAY_CODE = 1
HOUR = 10
scenario = pd.DataFrame({'order_dow': [DAY_CODE], 'order_hour_of_day': [HOUR]})
scenario['estimated_snapshot_orders'] = final_model.predict(make_features(scenario))
display(scenario)
print('Estimates are unconstrained regression outputs; inspect any negative value before use.')


## 11. Save model, metrics, split, and predictions

The ZIP contains the fitted model, metadata, package versions, feature-construction code,
test predictions, and CV/test scores. No Google credentials or spreadsheet URL are saved.
Only load joblib model files you trust. Use the saved package versions when reloading.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
bundle = {'model': final_model, 'feature_columns': X.columns.tolist(), 'target': TARGET,
          'model_name': selected_name, 'sklearn_version': sklearn.__version__}
joblib.dump(bundle, OUTPUT_DIR / 'model.joblib')
cv_results.to_csv(OUTPUT_DIR / 'cv_results.csv', index=False)
test_results.to_csv(OUTPUT_DIR / 'test_metrics.csv', index=False)
test_predictions.to_csv(OUTPUT_DIR / 'heldout_predictions.csv', index=False)
split_rows = df[INPUT_COLUMNS].copy()
split_rows['partition'] = 'train'
split_rows.loc[test_idx, 'partition'] = 'test'
split_rows.to_csv(OUTPUT_DIR / 'evaluation_split.csv', index=False)
if search is not None:
    pd.DataFrame(search.cv_results_).to_csv(OUTPUT_DIR / 'tuning_trials.csv', index=False)
elif (OUTPUT_DIR / 'tuning_trials.csv').exists():
    (OUTPUT_DIR / 'tuning_trials.csv').unlink()  # Remove stale results from a previous tuned run.

metadata = {
    'created_utc': datetime.now(timezone.utc).isoformat(), 'model': selected_name,
    'rows': len(df), 'data_sha256': data_fingerprint, 'worksheet': WORKSHEET_NAME,
    'random_state': RANDOM_STATE, 'test_fraction': TEST_FRACTION,
    'tuning_enabled': RUN_TUNING, 'search_iterations': N_ITER if RUN_TUNING else 0,
    'best_search_params': search.best_params_ if search is not None else None,
    'selected_parameters': {k: str(v) for k, v in final_model.get_params().items()},
    'test_hours': sorted(groups.iloc[test_idx].unique().tolist()),
    'evaluation': 'Hold out hour groups within one historical aggregate snapshot; not temporal forecasting.',
    'final_model': 'Refitted on all rows after holdout evaluation.',
}
(OUTPUT_DIR / 'metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
import importlib.metadata
packages = ['scikit-learn', 'pandas', 'numpy', 'scipy', 'joblib', 'matplotlib', 'openpyxl']
(OUTPUT_DIR / 'requirements.txt').write_text(
    '\n'.join(f'{p}=={importlib.metadata.version(p)}' for p in packages) + '\n', encoding='utf-8')


In [ ]:
feature_source = "import numpy as np\nimport pandas as pd\n\ndef make_features(frame):\n    day = pd.to_numeric(frame['order_dow'], errors='raise')\n    hour = pd.to_numeric(frame['order_hour_of_day'], errors='raise')\n    if day.isna().any() or hour.isna().any() or not day.between(0, 6).all() or not hour.between(0, 23).all():\n        raise ValueError('Day must be 0..6 and hour 0..23, without missing values.')\n    if (day != np.floor(day)).any() or (hour != np.floor(hour)).any():\n        raise ValueError('Day and hour must be integers.')\n    return pd.DataFrame({\n        'day_sin': np.sin(2 * np.pi * day / 7),\n        'day_cos': np.cos(2 * np.pi * day / 7),\n        'hour_sin': np.sin(2 * np.pi * hour / 24),\n        'hour_cos': np.cos(2 * np.pi * hour / 24),\n        'hour_sin_2': np.sin(4 * np.pi * hour / 24),\n        'hour_cos_2': np.cos(4 * np.pi * hour / 24),\n    }, index=frame.index)\n"
(OUTPUT_DIR / 'features.py').write_text(feature_source, encoding='utf-8')
reload_example = """import joblib
import pandas as pd
from features import make_features
bundle = joblib.load('model.joblib')
rows = pd.DataFrame({'order_dow': [1], 'order_hour_of_day': [10]})
X = make_features(rows)[bundle['feature_columns']]
print(bundle['model'].predict(X))
"""
(OUTPUT_DIR / 'predict.py').write_text(reload_example, encoding='utf-8')
archive_path = shutil.make_archive(str(OUTPUT_DIR), 'zip', OUTPUT_DIR)
print('Saved:', archive_path)


In [ ]:
DOWNLOAD_RESULTS = True
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(archive_path)


## Next project step

For a stronger customer/product reorder model, create one row per customer-product
at a historical cutoff, with features from earlier orders and a label from a later
order. Split by time (and customer where appropriate). Current Gold aggregates
cover the same observation period and cannot supply that prospective label safely.

## API references

- [gspread Colab authentication](https://docs.gspread.org/en/latest/advanced.html)
- [GroupShuffleSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html)
- [RandomizedSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html)
